# Si-Ge cluster expansion workflow - part 2

This is a CASM project tutorial to generate a phase diagram using a Si-Ge binary alloy cluster expansion fit to DFT calculations. The overall workflow is split into two parts.

Topics covered in part 2:

1. **Basis set construction**: Specify clusters and basis functions and construct a Clexulator
2. **Cluster expansion fitting**: Collect energies from import and mapping results and evaluate correlations, the per unitcell mean value of the symmetrically equivalent cluster functions. Fit coefficients to DFT calculated energies
4. **Monte Carlo simulations**: Run semi-grand canonical Monte Carlo simulations using the cluster expansion

** **Important** **: The instructions [here](https://prisms-center.github.io/CASMcode_pydocs/casm/bset/2.0/installation.html#environment-variable-configuration) describe how to set CASM environment variables before launching Jupyter.


## Setup

### Imports and paths

In [ ]:
import os
print(os.environ["CASM_PREFIX"])

In [ ]:
import pathlib
import numpy as np
import libcasm.xtal as xtal
import libcasm.configuration as casmconfig
from libcasm.xtal import pretty_json
from casm.project import Project
from casm.project.json_io import read_required, safe_dump

input_dir = pathlib.Path("input")
project_path = pathlib.Path("SiGe_occ")

# Check environment variables:
import os
print(f"CASM_PREFIX={os.environ["CASM_PREFIX"]}")

### Setup checks

- This notebook depends the import results obtained in part 1. Here we check that the necessary results exist. 

In [ ]:
# Construct project
project = Project.init(path=project_path)

# Enumeration and calculation types
enum_id = "occ_by_supercell.1"
calctype_id = "vasp-parameter-set-1"

enum = project.enum.get(enum_id)

# Calculations dir
target_dir = enum.calctype_dir(calctype_id)

# Imported structures
import_results_path = pathlib.Path(str(target_dir) + ".results.json")
import_results = read_required(import_results_path)

# Mapped configurations
mapped_structures_path = enum.enum_dir / f"mapping_results.{calctype_id}.json"
mapping_results = read_required(mapped_structures_path)

# Fitting data (formation energy, composition)
fitting_data_path = enum.enum_dir / f"fitting_data.{calctype_id}.json"
fitting_data = read_required(fitting_data_path)
formation_energy_per_unitcell = np.array(fitting_data.get("formation_energy_per_unitcell"))
comp_a = np.array(fitting_data.get("comp_a"))
relpath_list = fitting_data.get("relpath")

if len(mapping_results) != 214:
    raise ValueError(
        f"Expected 119 mapped structures, found {len(mapping_results)}. "
        "Try removing the SiGe_occ directory and re-running part 1."
    )


print(f"Found {len(mapping_results)} mapped structures")

## Basis set construction

### Basis set generating group

The formation energy is invariant under transformation by prim factor group operations, so we call it the "generating group" for our cluster expansion basis set. 

- [Project.sym]() gives quick access to symmetry information for the project.
- [Project.sym.print_factor_group]() gives a summary of the prim factor group operations.


In [ ]:
project.sym.print_factor_group()

### Specify clusters and basis functions

Use [*bset.make_bspecs*]() to construct a cluster expansion basis set for the Si-Ge formation energy. The parameters that may be useful are:

- *max_length*: The maximum site-to-site distance to allow in clusters, by number of sites in the cluster.
    - Example: ``max_length=[0.0, 0.0, 5.0, 4.0]`` specifies that pair clusters up to distance 5.0 and triplet clusters up to distance 4.0 should be included. The null cluster and point cluster values (elements 0 and 1) are arbitrary.
- *custom_generators*: Optionally, specify particular clusters to include regardless of *max_length*
- *occ_site_basis_functions_specs*: Select the occupation site basis functions. The most common options are:
  - "chebychev": An expansion (with correlation values all equal to 0) about the idealized random alloy where the probability of any of the allowed occupants on a particular site is the same.
  - "occupation": An expansion (with correlation values all equal to 0) about the default configuration where each site is occupied by the first allowed occupant in the Prim.occ_dof list.
  - See details [here](https://prisms-center.github.io/CASMcode_pydocs/casm/bset/2.0/usage/basis_function_specs.html#occupation-site-basis-functions)

In [ ]:
# Specify the basis set ID
# - Must be alphanumeric and underscores only
bset_id = "default"

# Specify maximum cluster site-to-site distance,
# by number of sites in the cluster
pair_max_length = 10.01
triplet_max_length = 7.27
quad_max_length = 4.0

# Use chebychev site basis functions (+x, -x)
occ_site_basis_functions_specs = "chebychev"

bset = project.bset.get(bset_id)
bset.make_bspecs(
    max_length=[
        0.0,  # null cluster, arbitrary
        0.0,  # point cluster, arbitrary
        pair_max_length,
        triplet_max_length,
        quad_max_length,
    ],
    occ_site_basis_functions_specs="chebychev",
)
bset.commit()

### The bspecs.json file

The previous steps created a "bspecs.json" file storing the basis set specifications:

- *cluster_specs*: specifications for which clusters to construct 
  cluster functions on 
- *basis_functions_specs*: specifications for which type of basis
  functions to generate
- *version*: specifies the Clexulator version to write (CASM v2+)

The "bspecs.json" file can also be edited manually, using the format described [here](https://prisms-center.github.io/CASMcode_docs/formats/casm/clex/ClexBasisSpecs/).


In [ ]:
bspecs_path = project.dir.bspecs(bset="default")
print(pretty_json(read_required(bspecs_path)))

### Generate and compile the Clexulator

The *bset.update* command generates and compiles a Clexulator (cluster expansion calculator).

Options include:

- *no_compile*: generate the basis set so that you can inspect the clusters and functions without compiling the Clexulator
- *only_compile*: re-compile a Clexulator using the existing files

In [ ]:
bset.update(
    # no_compile=False,
    # only_compile=False
)

### Inspect the cluster expansion

Print the cluster orbits:

- An "orbit" is the set of symmetrically equivalent objects. The "prototype" is one element in the orbit.
- In the context of periodic cluster expansion, the "multiplicity" of the orbit is the number of equivalent per unit cell (avoiding double counting clusters which include sites in multiple unit cell).
- The "cluster invariant group" is the set of prim factor group operations plus some lattice translation which leave the cluster unchanged.
  - This is the symmetry used to construct cluster functions. 

In [ ]:
bset.print_orbits(
    linear_orbit_indices=None,  # use i.e. set(range(1,5)) to print a subset
)

Print the cluster function prototypes:

- These are the cluster functions on the prototype cluster
- For a binary alloy like Si-Ge there is one function per cluster
  - For example, if the default occupation is Si, then the cluster expansion includes terms for Ge (point), Ge-Ge (pair), Ge-Ge-Ge (triplet), etc., but no Si-Ge (pair) term, because that is the same a Ge (point) term
- In general there may be >1 cluster per function, to account for interactions between different combinations of occupations 

In [ ]:
bset.display_occ_site_functions()

function_type = "orbit"

bset.display_functions(
    linear_function_indices=list(range(0, 5)),
    max_terms_per_line=4,
    function_type=function_type,
)

## Calculate correlations

### Correlations for ConfigurationSet

In [ ]:
bset_id = "default"
bset = project.bset.get(bset_id)

corr_calculator = bset.make_corr_calculator()
corr = corr_calculator.per_unitcell(enum.configuration_set)
print("corr:")
print(corr)
print("shape:", corr.shape)

### Calculate correlations for mapped configurations

In [ ]:
bset_id = "default"
bset = project.bset.get(bset_id)

supercell_set = enum.supercell_set

mapped_configs=[]
for relpath, mapping_data in mapping_results.items():
    mapped_config_with_properties = casmconfig.ConfigurationWithProperties.from_dict(
        data=mapping_data.get("mapped_configuration_with_properties"),
        supercells=supercell_set,
    )
    mapped_config = mapped_config_with_properties.configuration
    mapped_configs.append(mapped_config)

corr = corr_calculator.per_unitcell(mapped_configs)
print("corr:")
print(corr)
print("shape:", corr.shape)


## Fit coefficients

In [ ]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10

def plot_coeff(index, value, n):
    # Create a ColumnDataSource from the data
    source = ColumnDataSource(data=dict(x=index, y=value))
    
    # Create a new plot with a title and axis labels
    p = figure(height=350, title=f"{n} Coefficients",
               x_axis_label="Basis function index", y_axis_label="Coeff. Value",
               toolbar_location=None, tools="hover", tooltips="@x: @y")
    
    # Add a VBar renderer with a custom color map
    # factor_cmap maps colors to categories
    p.vbar(x='x', top='y', width=0.9, source=source,
           line_color='white')
    
    # # Customize the plot appearance
    p.x_range.start = 0
    p.x_range.end = 45
    
    # Show the results
    output_notebook()
    show(p)

In [ ]:
import libcasm.clexulator as casmclex
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

plot_list = []

def fit_n_coeff(n):
    X = corr
    y = formation_energy_per_unitcell
    
    # Construct LinearRegression estimator
    estimator = LinearRegression()
    
    # Construct RFE (Recursive feature elimination) feature selector
    # We want to select n features, and remove 1 feature at each step.
    n_features_to_select = n
    rfe_selector = RFE(
        estimator=estimator, 
        n_features_to_select=n_features_to_select, 
        step=1,
    )
    
    rfe_selector.fit(X, y)
    final_estimator = rfe_selector.estimator_
    
    # Get selected coefficients
    value = final_estimator.coef_
    index = np.where(rfe_selector.support_)[0]

    if hasattr(final_estimator, 'intercept_'):
        print(f"\nIntercept of the model fit on selected features: {final_estimator.intercept_:.4f}")
        index = np.array([0] + index.tolist(), dtype=int)
        value = np.array([final_estimator.intercept_] + value.tolist())
    
    # Print selected coefficients: index, value
    # for x in zip(index, value):
    #     print(x[0], x[1])
    
    plot_coeff(index, value, n)

    return casmclex.SparseCoefficients(
        index=index,
        value=value,
    )

def predict_ef(sparse_coeff, config):
    corr = corr_calculator.per_unitcell(config)
    return sparse_coeff * corr

def plot_energy(sparse_coeff):

    clex_formation_energy_per_unitcell = np.array([
        predict_ef(sparse_coeff, config) 
        for config in mapped_configs
    ])
        
    
    # Create a ColumnDataSource from the data
    source = ColumnDataSource(
        data=dict(
            relpath=[str(x) for x in relpath_list],
            comp_a=comp_a, 
            dft_Ef=formation_energy_per_unitcell,
            clex_Ef=clex_formation_energy_per_unitcell,
        )
    )

    tooltips = [
        ("relpath", "@relpath"),
        ("dft_Ef", "@dft_Ef"),
        ("clex_Ef", "@clex_Ef"),
        ("comp_a", "@comp_a"),
    ]
    
    # Create a new plot with a title and axis labels
    p = figure(height=350, title=f"Formation energy",
               x_axis_label="Parametric composition (a)", 
               y_axis_label="Formation energy / unitcell",
               tooltips=tooltips)
    
    p.scatter(
        "comp_a", "dft_Ef", source=source,
        marker="circle", fill_color=None, line_color="blue", size=10, alpha=0.5,
        legend_label="DFT",
    )
    p.scatter(
        "comp_a", "clex_Ef", source=source,
        color="red", size=6, alpha=0.5,
        legend_label="Predicted",
    )
    
    # Show the results
    output_notebook()
    show(p)

sparse_coeff = fit_n_coeff(5)
plot_energy(sparse_coeff)

sparse_coeff = fit_n_coeff(15)
plot_energy(sparse_coeff)

sparse_coeff = fit_n_coeff(25)
plot_energy(sparse_coeff)


## Monte Carlo simulations

### Load a Monte Carlo System

In [ ]:
from casm.project.json_io import read_required
from libcasm.clexmonte import (
    System,
)

system_data = read_required(input_dir / "system.json")
bset_dir = project.dir.bset_dir(bset=bset_id)

system = System.from_dict(
    data=system_data,
    search_path=[str(input_dir), str(bset_dir)],
)

### Run simulations

In [ ]:
from libcasm.clexmonte import MonteCalculator, make_initial_state

output_dir = project.path / "output"
output_dir.mkdir(parents=True, exist_ok=True)
summary_file = output_dir / "summary.json"

# construct a semi-grand canonical MonteCalculator
calculator = MonteCalculator(
    method="semigrand_canonical",
    system=system,
)

# construct default sampling fixture parameters
thermo = calculator.make_default_sampling_fixture_params(
    label="thermo",
    output_dir=str(output_dir),
)
print(xtal.pretty_json(thermo.to_dict()))

# construct the initial state (default configuration)
initial_state, motif, motif_id = make_initial_state(
    calculator=calculator,
    conditions={
        "temperature": 300.0,
        "param_chem_pot": [-1.0],
    },
    min_volume=1000,
)

# Run
sampling_fixture = calculator.run_fixture(
    state=initial_state,
    sampling_fixture_params=thermo,
)

In [ ]:
import numpy as np

# Run several, w/ dependent runs
mu_list = np.arange(-0.05, 0.05, step=0.01)

for temp in [100, 300, 600]:

    # construct the initial state (default configuration)
    initial_state, motif, motif_id = make_initial_state(
        calculator=calculator,
        conditions={
            "temperature": 300.0,
            "param_chem_pot": [-1.0],
        },
        min_volume=1000,
    )
    
    state = initial_state

    # construct default sampling fixture parameters
    output_dir = project.path / "mc" / f"output.{temp}"
    thermo = calculator.make_default_sampling_fixture_params(
        label="thermo",
        output_dir=str(output_dir),
    )
    
    for mu in mu_list:
        state.conditions.vector_values["param_chem_pot"] = [mu]
        sampling_fixture = calculator.run_fixture(
            state=state,
            sampling_fixture_params=thermo,
        )